# 🦀 Kafka Connect

![Kafka Connect PG](images/connect_pg.png)

Kafka Connect provides prebuilt connectors to integrate Kafka with other systems (sources and sinks). This notebook uses the **JDBC Sink Connector** to stream messages from a Kafka topic into the local PostgreSQL database running in Docker.

In [ ]:
from kafka import KafkaProducer
import json
import requests
from config.kafka_config import *

producer = KafkaProducer(
        **kafka_connection,
        value_serializer=lambda v: json.dumps(v).encode('ascii'),
        key_serializer=lambda v: json.dumps(v).encode('ascii')
    )

---

Let's create a new stream, adding the **schema** to it. 

Kafka Connect JDBC Sink requires a schema to be attached to the stream defining the its fields in detail. We have two choices:
* Attaching the schema to each JSON message
* Use schema registry with AVRO format

For the sake of this example we'll include the schema definition to the JSON message. Let's define the schema

In [ ]:
key_schema = {
    "type": "struct",
    "fields": [
        {
            "type": "int32",
            "optional": False,
            "field": "id"
        }
    ]
}

value_schema = {
    "type": "struct",
    "fields": [
        {
            "type": "string",
            "optional": False,
            "field": "name"
        },
        {
            "type": "string",
            "optional": False,
            "field": "pizza"}]
}

And send some data

In [ ]:
producer.send(
    topic_name+"_schema", 
    key={"schema": key_schema, "payload": {"id":1}},
    value={"schema": value_schema, 
           "payload": {"name":"👨 Frank", "pizza":"Margherita 🍕"}}
)

producer.send(
    topic_name+"_schema",
    key={"schema": key_schema, "payload": {"id":2}},
    value={"schema": value_schema, 
           "payload": {"name":"👨 Dan", "pizza":"Fries 🍕+🍟"}}
)


producer.send(
    topic_name+"_schema",
    key={"schema": key_schema, "payload": {"id":3}},
    value={"schema": value_schema,
           "payload": {"name":"👨 Jan", "pizza":"Mushrooms 🍕+🍄"}}
)

producer.flush()

## Start the Kafka Connect JDBC Sink Connector

Register the connector by POSTing its config to the local Kafka Connect REST API.

In [ ]:
connector_config = {
    "name": "sink_kafka_pg",
    "config": {
        "connector.class": "io.confluent.connect.jdbc.JdbcSinkConnector",
        "tasks.max": "1",
        "topics": topic_name + "_schema",
        "connection.url": "jdbc:postgresql://postgres:5432/postgres",
        "connection.user": pg_user,
        "connection.password": pg_pwd,
        "auto.create": "true",
        "value.converter": "org.apache.kafka.connect.json.JsonConverter",
        "key.converter": "org.apache.kafka.connect.json.JsonConverter",
        "value.converter.schemas.enable": "true",
        "key.converter.schemas.enable": "true",
        "insert.mode": "insert",
        "pk.mode": "none",
    }
}

response = requests.post(
    "http://kafka-connect:8083/connectors",
    headers={"Content-Type": "application/json"},
    data=json.dumps(connector_config)
)
print(response.status_code, response.json())

## Verify the Connector status

In [ ]:
response = requests.get("http://kafka-connect:8083/connectors/sink_kafka_pg/status")
print(response.json())

Let's add another **event**
with our **Python Producer**

In [ ]:
producer.send(
    topic_name+"_schema",
    key={
        "schema": key_schema,
        "payload": {"id":4}
    },
    value={
        "schema": value_schema,
        "payload": {"name":"👨 Giuseppe", "pizza":"Hawaii 🍕+🍍+🥓"}
          }
)


producer.flush()